# Resolve GBIF references

Match accepted species names for configured V3 organism groups that have at least one completed positive Phase 4 pathogen assessment against the GBIF Backbone, using the corresponding kingdom hint (Bacteria, Fungi, or Viruses). A link is recorded only for an exact canonical-name match at species rank; other GBIF matches remain diagnostics. Progress is checkpointed after every API batch.

By default, the notebook uses `run.organism_groups` from `configs/default.yaml` (currently `viruses`). It limits each selected group to species with at least one completed `candidate_pathogen_evidence=True` record in the Phase 4 pathogen decisions table. Change the group setting to `bacteria`, `fungi`, or any combination to resolve those pathogen candidates. Completed records are reused; set `FORCE_REFRESH = True` to repeat lookups for the selected groups.

In [1]:
from pathlib import Path
import os
import sys
import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pathogen_atlas_v3").is_dir():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise RuntimeError("Open this notebook from the Pathogen Atlas version3 repository.")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from pathogen_atlas_v3.common.io import load_yaml
from pathogen_atlas_v3.common.config import selected_organism_groups
CONFIG = load_yaml(PROJECT_ROOT / "configs/default.yaml")
ORGANISM_GROUPS = selected_organism_groups(CONFIG)
SPECIES_PATH = PROJECT_ROOT / CONFIG["paths"]["taxonomy_release"] / "species.parquet"
PATHOGEN_DECISIONS_PATH = PROJECT_ROOT / CONFIG["paths"]["release_root"] / "pathogen_decisions.parquet"
OUTPUT_PATH = PROJECT_ROOT / "data/06_references/release/species_gbif_links.csv"
FORCE_REFRESH = False

for required_path in (SPECIES_PATH, PATHOGEN_DECISIONS_PATH):
    if not required_path.exists():
        raise FileNotFoundError(f"Run Phases 1 and 4 first; required input is missing: {required_path}")

species = pd.read_parquet(SPECIES_PATH, columns=["organism_id", "organism_group", "accepted_name", "rank"])
decisions = pd.read_parquet(PATHOGEN_DECISIONS_PATH, columns=["organism_id", "organism_group", "candidate_pathogen_evidence", "status"])
pathogen_ids = set(decisions.loc[
    decisions.organism_group.isin(ORGANISM_GROUPS)
    & decisions.status.astype(str).eq("completed")
    & decisions.candidate_pathogen_evidence.fillna(False).astype(bool),
    "organism_id",
].astype(str))
scope = species.loc[
    species.organism_group.isin(ORGANISM_GROUPS)
    & species["rank"].astype(str).str.casefold().eq("species")
    & species.organism_id.astype(str).isin(pathogen_ids)
]
existing_count = 0
if OUTPUT_PATH.exists():
    previous = pd.read_csv(OUTPUT_PATH, keep_default_na=False)
    existing_count = int(previous.organism_group.isin(ORGANISM_GROUPS).sum()) if "organism_group" in previous else 0

display(Markdown(
    f"**Organism groups:** {', '.join(ORGANISM_GROUPS)}  ·  "
    f"**Species-rank records:** {scope.organism_id.nunique():,}  ·  "
    f"**Existing associations in scope:** {existing_count:,}  ·  "
    f"**Force refresh:** {FORCE_REFRESH}"
))


**Organism groups:** viruses  ·  **Species-rank records:** 257  ·  **Existing associations in scope:** 6,400  ·  **Force refresh:** False

## Resolve and save GBIF associations

The resolver writes a table keyed by `organism_id`, so identical accepted names in different domains remain separate. The same output file can be extended incrementally as you switch between bacteria, fungi, and viruses. Only the selected groups’ pathogen candidates are included.

In [2]:
from pathogen_atlas_v3.references.links import resolve_gbif_links

links = resolve_gbif_links(
    SPECIES_PATH,
    OUTPUT_PATH,
    ORGANISM_GROUPS,
    force_refresh=FORCE_REFRESH,
    organism_ids=pathogen_ids,
)
selected_links = links.loc[links.organism_group.isin(ORGANISM_GROUPS)]
found = selected_links.match_status.eq("found")
display(Markdown(
    f"**Saved:** `{OUTPUT_PATH.relative_to(PROJECT_ROOT)}`  ·  "
    f"**Records:** {len(selected_links):,}  ·  "
    f"**Exact GBIF species pages found:** {int(found.sum()):,}  ·  "
    f"**No exact page:** {int((~found).sum()):,}"
))
display(selected_links.head(12))


Resolving 247 records; reusing 10 completed rows.


Resolved 100/247 pending records.


Resolved 200/247 pending records.


Resolved 247/247 pending records.


Saved 257 associations to /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version3/data/06_references/release/species_gbif_links.csv


**Saved:** `data/06_references/release/species_gbif_links.csv`  ·  **Records:** 257  ·  **Exact GBIF species pages found:** 185  ·  **No exact page:** 72

,organism_id,organism_group,accepted_name,gbif_taxon_key,gbif_scientific_name,gbif_url,match_status,match_type,confidence
0,viruses:0399dd9c3f9db1845ece,viruses,Lyssavirus duvenhage,12265561,Lyssavirus duvenhage,https://www.gbif.org/species/12265561,found,EXACT,100
1,viruses:03dbb78f9fd8a55cc604,viruses,Orthonairovirus issykkulense,12254336,Orthonairovirus issykkulense,https://www.gbif.org/species/12254336,found,EXACT,100
2,viruses:0444a9e226e52ac1207d,viruses,Orthohepacivirus hominis,,,,not_found,NONE,100
3,viruses:04625a1bef3b7a0dbae2,viruses,Orthoflavivirus denguei,12228443,Orthoflavivirus denguei,https://www.gbif.org/species/12228443,found,EXACT,100
4,viruses:05d0996b7208bd22c57c,viruses,Orthobunyavirus tataguineense,12254929,Orthobunyavirus tataguineense,https://www.gbif.org/species/12254929,found,EXACT,100
5,viruses:070e5833c9c0eeec5b91,viruses,Orthopoxvirus akhmetapox,,,,not_found,NONE,100
6,viruses:075164ebd9df3fdc396b,viruses,Orthonairovirus erveense,12233024,Orthonairovirus erveense,https://www.gbif.org/species/12233024,found,EXACT,100
7,viruses:0c176a17061fe732e3bf,viruses,Alphapapillomavirus 14,10547972,Alphapapillomavirus 14,https://www.gbif.org/species/10547972,found,EXACT,100
8,viruses:10503db9ededcdb15c8f,viruses,Sunrhavirus garba,12193767,Sunrhavirus garba,https://www.gbif.org/species/12193767,found,EXACT,100
9,viruses:110f931a4f5b607e28d7,viruses,Morbillivirus hominis,12244494,Morbillivirus hominis,https://www.gbif.org/species/12244494,found,EXACT,100
